In [52]:
import sys

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_txt
from n_gram_tracing import (
    common_ngrams,
    filter_ngrams_with_outside_occurrences,
    tokenize_to_tokens,
    find_independent_token_ngram_spans
)

In [53]:
tokenizer, model = load_model("/Volumes/BCross/models/gpt2")

In [54]:
known_text = read_txt("../../data/282034.txt")
unknown_text = read_txt("../../data/1017882.txt")

In [96]:
known_text = (
    "i am walking through the city at night while the rain falls heavily outside. "
    "the old man said i am tired of this weather and i am going home soon. "
    "later i see therefore i am becomes part of the discussion during dinner. "
    "my friend laughed and said i am dead inside after the long meeting yesterday. "
    "near the end of the evening i am excited for tomorrow because the journey finally begins. "
    "the teacher repeated that i am responsible for my own decisions and i am expected to learn quickly. "
    "before leaving the restaurant i am thinking carefully about the strange conversation we had earlier."
)

unknown_text = (
    "i am walking through the garden at night while the cold wind moves slowly outside. "
    "the young woman said i am tired of this weather and i am staying home tonight. "
    "later i see, therefore i am becomes part of the debate during class. "
    "my brother laughed and said i am dead inside after the difficult interview yesterday. "
    "near the end of the afternoon i am excited for tomorrow because the adventure finally begins. "
    "the manager repeated that i am responsible for my own choices so i am found to improve quickly. "
    "before leaving the building i am wondering carefully about the strange conversation we had earlier."
)

In [97]:
# known_text = "I am a cat. I see therefore i am. I see, i am"
# unknown_text = "I am the dog. I see therefore i am. I am what i see i am"

In [98]:
import time

start_time = time.perf_counter()
common = common_ngrams(
    text1=known_text,
    text2=unknown_text,
    tokenizer=tokenizer,
    include_subgrams=True,
    lowercase=True
)

end_time = time.perf_counter()
print(f"Time taken: {end_time - start_time:.4f} seconds")

Time taken: 0.0115 seconds


In [99]:
filtered = filter_ngrams_with_outside_occurrences(
    ngrams=common,
    text=known_text,
    tokenizer=tokenizer,
    lowercase=True,
)

filtered

[['Ġduring'],
 ['Ġhome'],
 ['Ġto'],
 ['.', 'Ġmy'],
 ['Ġi', 'Ġam'],
 ['Ġand', 'Ġi', 'Ġam'],
 ['Ġoutside', '.', 'Ġthe'],
 ['.', 'Ġlater', 'Ġi', 'Ġsee'],
 ['Ġat', 'Ġnight', 'Ġwhile', 'Ġthe'],
 ['Ġfinally', 'Ġbegins', '.', 'Ġthe'],
 ['i', 'Ġam', 'Ġwalking', 'Ġthrough', 'Ġthe'],
 ['Ġquickly', '.', 'Ġbefore', 'Ġleaving', 'Ġthe'],
 ['Ġi', 'Ġam', 'Ġexcited', 'Ġfor', 'Ġtomorrow', 'Ġbecause', 'Ġthe'],
 ['Ġtherefore', 'Ġi', 'Ġam', 'Ġbecomes', 'Ġpart', 'Ġof', 'Ġthe'],
 ['Ġyesterday', '.', 'Ġnear', 'Ġthe', 'Ġend', 'Ġof', 'Ġthe'],
 ['Ġrepeated', 'Ġthat', 'Ġi', 'Ġam', 'Ġresponsible', 'Ġfor', 'Ġmy', 'Ġown'],
 ['Ġcarefully',
  'Ġabout',
  'Ġthe',
  'Ġstrange',
  'Ġconversation',
  'Ġwe',
  'Ġhad',
  'Ġearlier',
  '.'],
 ['Ġlaughed',
  'Ġand',
  'Ġsaid',
  'Ġi',
  'Ġam',
  'Ġdead',
  'Ġinside',
  'Ġafter',
  'Ġthe'],
 ['Ġsaid',
  'Ġi',
  'Ġam',
  'Ġtired',
  'Ġof',
  'Ġthis',
  'Ġweather',
  'Ġand',
  'Ġi',
  'Ġam']]

In [91]:
tokens = tokenize_to_tokens(
    unknown_text,
    tokenizer=tokenizer,
    lowercase=True,
)

In [100]:
spans = find_independent_token_ngram_spans(
    tokens,
    ['.', 'Ġlater', 'Ġi', 'Ġsee'],
    filtered,
)

[
    tokens_to_text(tokens[:end], tokenizer)
    for start, end in spans
]

['i am walking through the garden at night while the cold wind moves slowly outside. the young woman said i am tired of this weather and i am staying home tonight. later i see']

In [90]:
spans

[(94, 95)]

In [93]:
def check_subgram_relationships(ngrams):
    """
    For each n-gram, report which longer n-grams contain it.
    """
    grams = list(dict.fromkeys(tuple(g) for g in ngrams))

    rows = []

    for child in grams:
        parents = []

        for parent in grams:
            if len(parent) <= len(child):
                continue

            for i in range(len(parent) - len(child) + 1):
                if parent[i:i + len(child)] == child:
                    parents.append({
                        "parent_ngram": list(parent),
                        "offset": i,
                    })

        rows.append({
            "ngram": list(child),
            "num_tokens": len(child),
            "is_subgram": len(parents) > 0,
            "num_parents": len(parents),
            "parents": parents,
        })

    return rows

In [94]:
check_subgram_relationships(filtered)

[{'ngram': ['Ġduring'],
  'num_tokens': 1,
  'is_subgram': False,
  'num_parents': 0,
  'parents': []},
 {'ngram': ['Ġhome'],
  'num_tokens': 1,
  'is_subgram': False,
  'num_parents': 0,
  'parents': []},
 {'ngram': ['Ġto'],
  'num_tokens': 1,
  'is_subgram': False,
  'num_parents': 0,
  'parents': []},
 {'ngram': ['.', 'Ġmy'],
  'num_tokens': 2,
  'is_subgram': False,
  'num_parents': 0,
  'parents': []},
 {'ngram': ['Ġi', 'Ġam'],
  'num_tokens': 2,
  'is_subgram': True,
  'num_parents': 7,
  'parents': [{'parent_ngram': ['Ġand', 'Ġi', 'Ġam'], 'offset': 1},
   {'parent_ngram': ['Ġi',
     'Ġam',
     'Ġexcited',
     'Ġfor',
     'Ġtomorrow',
     'Ġbecause',
     'Ġthe'],
    'offset': 0},
   {'parent_ngram': ['Ġrepeated',
     'Ġthat',
     'Ġi',
     'Ġam',
     'Ġresponsible',
     'Ġfor',
     'Ġmy',
     'Ġown'],
    'offset': 2},
   {'parent_ngram': ['Ġlaughed',
     'Ġand',
     'Ġsaid',
     'Ġi',
     'Ġam',
     'Ġdead',
     'Ġinside',
     'Ġafter',
     'Ġthe'],
    'of